# 02 Pipeline Development - Semantic Ranking
Start building the ranking pipeline for the full 100k candidates.

In [ ]:
%pip install pandas numpy tqdm python-docx sentence-transformers scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import os
import json
import re
import pandas as pd
import numpy as np
from tqdm import tqdm
from docx import Document
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from IPython.display import display

base_dir = "/content/ai-candidate-ranking"
if not os.path.exists(base_dir):
    base_dir = ".." if os.path.basename(os.getcwd()) == "notebooks" else "." # Local fallback
    
raw_dir = os.path.join(base_dir, "data", "raw")

### 1. Streaming Data Loader
Yield candidate dicts line by line.

In [ ]:
def load_candidates(path, limit=None):
    """
    Generator to yield candidate dicts line by line from JSONL.
    """
    count = 0
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            yield json.loads(line)
            count += 1
            if limit and count >= limit:
                break

### 2. Build Candidate Text
Concatenate headline, summary, last 2 roles, and up to 20 skills (prioritizing AI skills).

In [ ]:
ai_keywords = [
    "machine learning", "deep learning", "nlp", "search", "retrieval", 
    "embedding", "vector", "pytorch", "tensorflow", "mlflow", "llm", "ai",
    "artificial intelligence", "recommender", "recommendation"
]
ai_pattern = re.compile("|".join(ai_keywords), re.IGNORECASE)

def build_candidate_text(candidate):
    """
    Returns a single string representation of the candidate.
    """
    profile = candidate.get("profile", {})
    history = candidate.get("career_history", [])
    skills = candidate.get("skills", [])
    
    parts = []
    
    # Profile text
    headline = profile.get("headline")
    if headline: parts.append(f"Headline: {headline}")
    
    summary = profile.get("summary")
    if summary: parts.append(f"Summary: {summary}")
        
    # Last 2 roles
    # Assuming the list is ordered newest to oldest
    for role in history[:2]:
        title = role.get("title")
        desc = role.get("description")
        if title or desc:
            role_str = f"Role: {title or ''} - {desc or ''}"
            parts.append(role_str)
            
    # Skills (up to 20, AI prioritized)
    skill_names = [s.get("name", str(s)) if isinstance(s, dict) else str(s) for s in skills]
    
    ai_skills = [s for s in skill_names if ai_pattern.search(s)]
    other_skills = [s for s in skill_names if not ai_pattern.search(s)]
    
    top_skills = (ai_skills + other_skills)[:20]
    if top_skills:
        parts.append("Skills: " + ", ".join(top_skills))
        
    return " | ".join(parts)

### 3. Build Job Description Text
Extract important sections from `job_description.docx`.

In [ ]:
def build_jd_text(docx_path):
    """
    Loads the JD and extracts critical sections to form a concise query string.
    """
    try:
        doc = Document(docx_path)
        paragraphs = [p.text.strip() for p in doc.paragraphs if p.text.strip()]
        full_text = "\n".join(paragraphs)
        
        # In a real scenario, we'd use regex or keyword finding to slice specific 
        # sections (e.g., 'Things you absolutely need', 'Things we explicitly do NOT want').
        # If the exact headings aren't predictable, returning the full text is the safest 
        # fallback for generating a comprehensive embedding.
        
        jd_text = full_text
        return jd_text
    except Exception as e:
        print(f"Error reading JD: {e}")
        return ""

### 4. Load Sentence Transformer
Using `sentence-transformers/all-MiniLM-L6-v2` loaded once for reuse.

In [ ]:
model_name = "sentence-transformers/all-MiniLM-L6-v2"
print(f"Loading model: {model_name}...")
model = SentenceTransformer(model_name)
print("Model successfully loaded!")

Loading model: sentence-transformers/all-MiniLM-L6-v2...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

### 5. Generate Embeddings and Compute Similarity
Stream candidates, encode text, encode JD, and compute cosine similarities.

In [ ]:
# Try to load jsonl first, fallback to json array if running locally with sample data
cands_path = os.path.join(raw_dir, "candidates.jsonl")
is_json_array = False

if not os.path.exists(cands_path):
    cands_path = os.path.join(raw_dir, "sample_candidates.json")
    is_json_array = True
    
candidates_list = []
candidate_texts = []

print(f"Loading candidates from {cands_path}...")
if is_json_array:
    # Fallback for small sample arrays
    with open(cands_path, "r") as f:
        content = f.read().strip()
        if content:
            data = json.loads(content)
            for cand in data[:5000]:
                candidates_list.append(cand)
                candidate_texts.append(build_candidate_text(cand))
else:
    for cand in load_candidates(cands_path, limit=5000):
        candidates_list.append(cand)
        candidate_texts.append(build_candidate_text(cand))

print(f"Extracted {len(candidates_list)} candidates for development.")

if candidate_texts:
    print("Encoding candidate texts...")
    cand_embeddings = model.encode(candidate_texts, batch_size=32, show_progress_bar=True)
else:
    cand_embeddings = np.array([])

jd_path = os.path.join(raw_dir, "job_description.docx")
jd_text = build_jd_text(jd_path)

if jd_text:
    print("\nEncoding JD...")
    jd_embedding = model.encode([jd_text])
else:
    jd_embedding = np.array([])

similarities = []
if cand_embeddings.shape[0] > 0 and jd_embedding.shape[0] > 0:
    print("Computing cosine similarities...")
    similarities = cosine_similarity(jd_embedding, cand_embeddings)[0]


Loading candidates from ../data/raw/candidates.jsonl...
Extracted 5000 candidates for development.
Encoding candidate texts...


Batches:   0%|          | 0/157 [00:00<?, ?it/s]


Encoding JD...
Computing cosine similarities...


### 6. Create `dev_semantic_df`
Aggregate results into a DataFrame.

In [ ]:
records = []
for i, cand in enumerate(candidates_list):
    cid = cand.get("candidate_id")
    profile = cand.get("profile", {})
    signals = cand.get("redrob_signals", {})
    skills = cand.get("skills", [])
    
    ai_skills_count = sum(1 for s in skills if ai_pattern.search(s.get("name", str(s)) if isinstance(s, dict) else str(s)))
    
    records.append({
        "candidate_id": cid,
        "semantic_similarity": similarities[i] if len(similarities) > i else 0.0,
        "years_of_experience": profile.get("years_of_experience"),
        "current_title": profile.get("current_title"),
        "current_industry": profile.get("current_industry"),
        "ai_core_skills_count": ai_skills_count,
        "recruiter_response_rate": signals.get("recruiter_response_rate")
    })

dev_semantic_df = pd.DataFrame(records)
display(dev_semantic_df.head())

,candidate_id,semantic_similarity,years_of_experience,current_title,current_industry,ai_core_skills_count,recruiter_response_rate
0,CAND_0000001,0.497641,6.9,Backend Engineer,IT Services,3,0.34
1,CAND_0000002,0.584883,12.5,Operations Manager,IT Services,0,0.29
2,CAND_0000003,0.550712,1.1,Customer Support,IT Services,0,0.46
3,CAND_0000004,0.530352,3.8,Marketing Manager,Paper Products,1,0.26
4,CAND_0000005,0.486976,11.0,Accountant,Manufacturing,1,0.37


### 7. View Top 30 Candidates
Inspect whether they look plausible for the Senior AI Engineer role.

In [ ]:
print("=== Top 30 Candidates by Semantic Similarity ===")
top_30 = dev_semantic_df.sort_values(by="semantic_similarity", ascending=False).head(30)
display(top_30)

=== Top 30 Candidates by Semantic Similarity ===


,candidate_id,semantic_similarity,years_of_experience,current_title,current_industry,ai_core_skills_count,recruiter_response_rate
321,CAND_0000322,0.655480,12.0,HR Manager,Manufacturing,6,0.11
3961,CAND_0003962,0.654567,13.0,HR Manager,Software,4,0.08
2661,CAND_0002662,0.654281,5.4,Operations Manager,IT Services,9,0.64
3066,CAND_0003067,0.653597,3.9,HR Manager,IT Services,6,0.61
3675,CAND_0003676,0.652920,13.8,Mechanical Engineer,Manufacturing,7,0.62
4192,CAND_0004193,0.652278,2.2,HR Manager,IT Services,7,0.63
4329,CAND_0004330,0.647980,13.8,Mechanical Engineer,Software,6,0.74
846,CAND_0000847,0.647534,14.5,HR Manager,Software,7,0.70
2450,CAND_0002451,0.646465,12.6,Civil Engineer,Software,5,0.26
4910,CAND_0004911,0.645671,13.9,Marketing Manager,Manufacturing,9,0.17


### 8. Feature Engineering for Ranking
Engineer richer recruiter-style features for each candidate to build a comprehensive feature vector.

In [ ]:
import datetime
from dateutil import parser

# Reference date for recency calculations
REF_DATE = datetime.datetime(2025, 12, 31)

ai_titles = ["machine learning", "ml engineer", "data scientist", "applied scientist", 
             "ai engineer", "search engineer", "relevance engineer", "recommendation"]
ai_title_pattern = re.compile("|".join(ai_titles), re.IGNORECASE)

consulting_companies = ["infosys", "wipro", "tcs", "accenture", "cognizant", "capgemini", "ibm"]
consulting_pattern = re.compile("|".join(consulting_companies), re.IGNORECASE)

def calc_role_title_score(title):
    if not title: return 0.0
    if ai_title_pattern.search(title): return 1.0
    non_ml = ["marketing", "operations", "project manager", "hr", "sales"]
    if any(n in title.lower() for n in non_ml): return -0.5
    return 0.2

def calc_product_company_score(company, industry):
    score = 0.5
    comb = str(company).lower() + " " + str(industry).lower()
    if consulting_pattern.search(comb): return 0.0
    if "product" in comb or "tech" in comb or "software" in comb: return 0.8
    return score

def calc_experience_band_match(yoe):
    if yoe is None: return 0.0
    try: y = float(yoe)
    except: return 0.0
    if 6 <= y <= 8: return 1.0
    if 4 <= y < 6 or 8 < y <= 10: return 0.8
    if 10 < y <= 12: return 0.5
    return 0.2

def calc_ml_years_estimate(history):
    ml_years = 0.0
    for role in history:
        title = role.get("title", "")
        start = role.get("start_date")
        end = role.get("end_date")
        if ai_title_pattern.search(title) and start and end:
            try:
                s = parser.parse(start)
                e = parser.parse(end) if end.lower() != "present" else REF_DATE
                months = (e.year - s.year) * 12 + (e.month - s.month)
                ml_years += max(0, months / 12.0)
            except: pass
    return ml_years

def calc_ai_skill_depth(skills):
    prof_weights = {"beginner": 0.2, "intermediate": 0.5, "advanced": 0.8, "expert": 1.0}
    depth = 0.0
    for s in skills:
        name = s.get("name", "") if isinstance(s, dict) else str(s)
        if ai_pattern.search(name) and isinstance(s, dict):
            prof = s.get("proficiency", "beginner").lower()
            dur = s.get("duration_months", 0)
            w = prof_weights.get(prof, 0.2)
            dur_score = min(float(dur) / 60.0, 1.0)
            depth += (w * 0.5 + dur_score * 0.5)
    return depth

def calc_availability_score(signals):
    open_flag = 1.0 if signals.get("open_to_work_flag") else 0.0
    recency_score = 0.0
    last_active = signals.get("last_active_date")
    if last_active:
        try:
            la_date = parser.parse(last_active)
            days_ago = (REF_DATE - la_date.replace(tzinfo=None)).days
            recency_score = max(0.0, 1.0 - (days_ago / 180.0))
        except: pass
    resp_rate = float(signals.get("recruiter_response_rate", 0))
    prof_comp = float(signals.get("profile_completeness_score", 0))
    apps_30d = min(float(signals.get("applications_submitted_30d", 0)) / 10.0, 1.0)
    return (open_flag * 0.2) + (recency_score * 0.3) + (resp_rate * 0.3) + (prof_comp * 0.1) + (apps_30d * 0.1)

def calc_reliability_score(signals):
    int_comp = float(signals.get("interview_completion_rate", 0))
    offer_acc = float(signals.get("offer_acceptance_rate", -1))
    v_email = 1.0 if signals.get("verified_email") else 0.0
    v_phone = 1.0 if signals.get("verified_phone") else 0.0
    v_link = 1.0 if signals.get("linkedin_connected") else 0.0
    verification_score = (v_email + v_phone + v_link) / 3.0
    
    if offer_acc != -1:
        return (int_comp * 0.5) + (offer_acc * 0.3) + (verification_score * 0.2)
    return (int_comp * 0.7) + (verification_score * 0.3)

def calc_geo_fit(location):
    if not location: return 0.5
    loc = location.lower()
    if "pune" in loc or "noida" in loc: return 1.0
    if "india" in loc: return 0.8
    if "us" in loc or "usa" in loc or "uk" in loc: return 0.2
    return 0.5
    
def calc_notice_penalty(days):
    if days is None: return 0.0
    try:
        d = float(days)
        if d <= 30: return 0.0
        if d <= 60: return 0.2
        if d <= 90: return 0.6
        return 1.0
    except: return 0.0

def calc_honeypot_risk(yoe, hist_years, skills, ml_years):
    risk = 0.0
    if hist_years > yoe + 3: risk += 0.3
    
    expert_zero_dur = sum(1 for s in skills if isinstance(s, dict) and ai_pattern.search(s.get("name","")) 
                          and s.get("proficiency", "").lower() in ["advanced", "expert"] 
                          and float(s.get("duration_months", 0)) <= 1)
    if expert_zero_dur > 2: risk += 0.4
    
    ai_skills_count = sum(1 for s in skills if ai_pattern.search(s.get("name", str(s)) if isinstance(s, dict) else str(s)))
    if ai_skills_count > 10 and ml_years < 0.5: risk += 0.4
        
    return min(risk, 1.0)

### 9. Build `dev_features_df`
Apply the logic across the dev candidate subset.

In [ ]:
feature_records = []

# Use the previously populated dev_semantic_df for semantic similarities
semantic_sim_dict = dict(zip(dev_semantic_df['candidate_id'], dev_semantic_df['semantic_similarity']))

for cand in tqdm(candidates_list, desc="Engineering Features"):
    cid = cand.get("candidate_id")
    profile = cand.get("profile", {})
    history = cand.get("career_history", [])
    skills = cand.get("skills", [])
    signals = cand.get("redrob_signals", {})
    
    role_score = calc_role_title_score(profile.get("current_title"))
    company_score = calc_product_company_score(profile.get("current_company"), profile.get("current_industry"))
    
    yoe = profile.get("years_of_experience", 0)
    try: yoe_float = float(yoe) if yoe else 0.0
    except: yoe_float = 0.0
    
    exp_band_match = calc_experience_band_match(yoe)
    ml_years = calc_ml_years_estimate(history)
    
    ai_skills_count = sum(1 for s in skills if ai_pattern.search(s.get("name", str(s)) if isinstance(s, dict) else str(s)))
    ai_depth_score = calc_ai_skill_depth(skills)
    
    avail_score = calc_availability_score(signals)
    rel_score = calc_reliability_score(signals)
    gh_score = max(0, float(signals.get("github_activity_score", 0)))
    
    geo_score = calc_geo_fit(profile.get("location"))
    pref_work = profile.get("preferred_work_mode", "").lower()
    wm_fit = 1.0 if "hybrid" in pref_work or "flexible" in pref_work else 0.5
    notice_penalty = calc_notice_penalty(profile.get("notice_period_days"))
    
    hist_years = 0
    if history:
        try:
            earliest = min([parser.parse(r["start_date"]) for r in history if "start_date" in r])
            hist_years = (REF_DATE - earliest.replace(tzinfo=None)).days / 365.25
        except: pass
            
    honeypot_risk = calc_honeypot_risk(yoe_float, hist_years, skills, ml_years)
    
    feature_records.append({
        "candidate_id": cid,
        "semantic_similarity": semantic_sim_dict.get(cid, 0.0),
        "role_title_score": role_score,
        "product_company_score": company_score,
        "total_years_experience": yoe_float,
        "experience_band_match": exp_band_match,
        "ml_years_estimate": ml_years,
        "ai_core_skills_count": ai_skills_count,
        "ai_skill_depth_score": ai_depth_score,
        "availability_score": avail_score,
        "reliability_score": rel_score,
        "github_fit_score": gh_score,
        "geo_fit_score": geo_score,
        "work_mode_fit": wm_fit,
        "notice_period_penalty": notice_penalty,
        "honeypot_risk_score": honeypot_risk
    })

dev_features_df = pd.DataFrame(feature_records)

Engineering Features: 100%|██████████| 5000/5000 [00:00<00:00, 11343.12it/s]


### 10. Summary Statistics & Review
Display summary statistics and the top rows of the engineered features.

In [ ]:
print("=== Feature Summary Statistics ===")
display(dev_features_df.describe())

print("\n=== Top Rows of Engineered Features ===")
display(dev_features_df.head(10))

=== Feature Summary Statistics ===


,semantic_similarity,role_title_score,product_company_score,total_years_experience,experience_band_match,ml_years_estimate,ai_core_skills_count,ai_skill_depth_score,availability_score,reliability_score,github_fit_score,geo_fit_score,work_mode_fit,notice_period_penalty,honeypot_risk_score
count,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.0,5000.0,5000.0
mean,0.514482,0.003300,0.481300,7.110900,0.570100,0.018483,0.839000,0.378242,6.287393,0.585918,10.162820,0.535360,0.5,0.0,0.0
std,0.040348,0.327751,0.311608,3.800649,0.325074,0.246942,1.581891,0.741833,1.747620,0.137154,17.443823,0.148816,0.0,0.0,0.0
min,0.351789,-0.500000,0.000000,1.000000,0.200000,0.000000,0.000000,0.000000,2.814000,0.203000,0.000000,0.200000,0.5,0.0,0.0
25%,0.490613,-0.500000,0.000000,3.900000,0.200000,0.000000,0.000000,0.000000,4.805583,0.488000,0.000000,0.500000,0.5,0.0,0.0
50%,0.514770,0.200000,0.500000,6.800000,0.800000,0.000000,0.000000,0.000000,6.352167,0.585000,0.000000,0.500000,0.5,0.0,0.0
75%,0.537589,0.200000,0.800000,9.900000,0.800000,0.000000,1.000000,0.458333,7.738750,0.683000,16.200000,0.500000,0.5,0.0,0.0
max,0.655480,1.000000,0.800000,15.000000,1.000000,5.083333,10.000000,7.308333,10.741000,0.979000,96.900000,1.000000,0.5,0.0,0.0



=== Top Rows of Engineered Features ===


,candidate_id,semantic_similarity,role_title_score,product_company_score,total_years_experience,experience_band_match,ml_years_estimate,ai_core_skills_count,ai_skill_depth_score,availability_score,reliability_score,github_fit_score,geo_fit_score,work_mode_fit,notice_period_penalty,honeypot_risk_score
0,CAND_0000001,0.497641,0.2,0.5,6.9,1.0,0.0,3,1.675000,9.545333,0.662333,9.2,0.5,0.5,0.0,0.0
1,CAND_0000002,0.584883,-0.5,0.0,12.5,0.2,0.0,0,0.000000,8.385333,0.434000,0.0,0.5,0.5,0.0,0.0
2,CAND_0000003,0.550712,0.2,0.0,1.1,0.2,0.0,0,0.000000,3.851333,0.702000,0.0,0.2,0.5,0.0,0.0
3,CAND_0000004,0.530352,-0.5,0.8,3.8,0.2,0.0,1,0.475000,3.458000,0.545000,0.0,0.5,0.5,0.0,0.0
4,CAND_0000005,0.486976,0.2,0.5,11.0,0.5,0.0,1,0.541667,8.939333,0.718000,0.0,0.5,0.5,0.0,0.0
5,CAND_0000006,0.542137,0.2,0.5,6.0,1.0,0.0,0,0.000000,3.484333,0.599000,0.0,0.2,0.5,0.0,0.0
6,CAND_0000007,0.542803,0.2,0.0,5.5,0.8,0.0,0,0.000000,8.197667,0.629000,0.0,0.5,0.5,0.0,0.0
7,CAND_0000008,0.558353,-0.5,0.0,3.6,0.2,0.0,0,0.000000,6.746000,0.618000,0.0,1.0,0.5,0.0,0.0
8,CAND_0000009,0.566052,0.2,0.8,11.0,0.5,0.0,0,0.000000,4.544000,0.547333,0.0,0.5,0.5,0.0,0.0
9,CAND_0000010,0.522668,0.2,0.5,4.6,0.8,0.0,1,0.391667,8.878333,0.571000,33.7,0.5,0.5,0.0,0.0


### 11. Final Scoring Heuristic
Combine semantic similarity with all engineered features to compute the `final_score`.

In [ ]:
# Define weights for the heuristic scoring
weights = {
    "semantic": 0.35,
    "role": 0.10,
    "product": 0.10,
    "exp_band": 0.10,
    "ml_years": 0.10,
    "ai_depth": 0.05,
    "avail": 0.05,
    "rel": 0.05,
    "github": 0.05,
    "geo": 0.03,
    "work_mode": 0.02
}

# Penalties
alpha = 0.5 # Notice period penalty scale
beta = 1.0  # Honeypot penalty scale

# Normalize ML years
max_ml_years = dev_features_df["ml_years_estimate"].max()
if max_ml_years == 0: max_ml_years = 1.0

def compute_final_score(row):
    score = (
        row["semantic_similarity"] * weights["semantic"] +
        row["role_title_score"] * weights["role"] +
        row["product_company_score"] * weights["product"] +
        row["experience_band_match"] * weights["exp_band"] +
        (row["ml_years_estimate"] / max_ml_years) * weights["ml_years"] +
        min(row["ai_skill_depth_score"], 1.0) * weights["ai_depth"] +
        row["availability_score"] * weights["avail"] +
        row["reliability_score"] * weights["rel"] +
        row["github_fit_score"] * weights["github"] +
        row["geo_fit_score"] * weights["geo"] +
        row["work_mode_fit"] * weights["work_mode"]
    )
    
    # Apply penalties
    score -= (row["notice_period_penalty"] * alpha)
    score -= (row["honeypot_risk_score"] * beta)
    
    # Cap between 0 and 1
    return max(0.0, min(1.0, score))

dev_features_df["final_score"] = dev_features_df.apply(compute_final_score, axis=1)

# Sort descending
ranked_candidates = dev_features_df.sort_values(by="final_score", ascending=False)

### 12. Review Top Candidates
Ensure they have relevant AI/ML experience, come from product backgrounds, are active, and are not honeypots.

In [ ]:
top_50 = ranked_candidates.head(50).copy()

print("=== Top 50 Ranked Candidates Summary ===")
display(top_50[["candidate_id", "final_score", "semantic_similarity", "total_years_experience", "ml_years_estimate", "honeypot_risk_score", "availability_score"]].head(15))

=== Top 50 Ranked Candidates Summary ===


,candidate_id,final_score,semantic_similarity,total_years_experience,ml_years_estimate,honeypot_risk_score,availability_score
0,CAND_0000001,1.0,0.497641,6.9,0.0,0.0,9.545333
2483,CAND_0002484,1.0,0.512188,1.3,0.0,0.0,6.677333
2437,CAND_0002438,1.0,0.523374,5.7,0.0,0.0,7.418667
2438,CAND_0002439,1.0,0.498999,6.0,0.0,0.0,7.171667
2439,CAND_0002440,1.0,0.497602,8.1,0.0,0.0,9.371000
2440,CAND_0002441,1.0,0.620190,10.2,0.0,0.0,4.628667
2441,CAND_0002442,1.0,0.521828,4.1,0.0,0.0,9.311667
2447,CAND_0002448,1.0,0.566945,6.1,0.0,0.0,7.101333
2450,CAND_0002451,1.0,0.646465,12.6,0.0,0.0,6.191333
2452,CAND_0002453,1.0,0.525965,4.1,0.0,0.0,8.611667


### 13. Explanability: Reasoning Generation
Generate 1-2 sentences of reasoning for the ranking, referencing their title, skills, JD alignment, and signals.

In [ ]:
def generate_reasoning(row):
    # Lookup full candidate context
    cand = next((c for c in candidates_list if c["candidate_id"] == row["candidate_id"]), None)
    if not cand:
        return "Candidate details not found."
    
    title = cand.get("profile", {}).get("Professional")
    yoe = cand.get("profile", {}).get("years_of_experience", 0)
    skills = cand.get("skills", [])
    signals = cand.get("redrob_signals", {})
    
    # Extract AI skills
    skill_names = [s.get("name", str(s)) if isinstance(s, dict) else str(s) for s in skills]
    ai_skills_present = [s for s in skill_names if ai_pattern.search(s)]
    named_skills = ai_skills_present[:3]
    
    # JD Alignment point
    sim = row["semantic_similarity"]
    if sim > 0.6:
        alignment = "exceptional alignment with JD requirements for semantic ranking and search"
    elif sim > 0.4:
        alignment = "strong background in machine learning and relevant embeddings"
    else:
        alignment = "some exposure to ML/AI tooling"
        
    # Behavioral signal
    try: resp_rate = float(signals.get("recruiter_response_rate", 0))
    except: resp_rate = 0.0
    
    try: notice = float(cand.get("profile", {}).get("notice_period_days", 0))
    except: notice = 0.0
        
    last_act = signals.get("last_active_date", "")
    
    if resp_rate > 0.8:
        behavior = f"Highly responsive to recruiters ({int(resp_rate*100)}% rate)"
    elif last_act:
        behavior = f"Recently active on the platform ({last_act[:10]})"
    else:
        behavior = "Moderate platform engagement"
        
    # Concatenate sentence
    sentence1 = f"An experienced {title} with {yoe} years of experience, demonstrating {alignment}."
    skills_str = f" Includes core AI competencies like {', '.join(named_skills)} among {row['ai_core_skills_count']} total ML skills." if named_skills else " Lacks explicitly named AI/ML tools."
    sentence2 = f" {behavior}."
    
    # Add concerns if applicable
    concern = ""
    if row["product_company_score"] < 0.3:
        concern = " However, their background leans heavily towards IT services rather than product."
    elif row["notice_period_penalty"] > 0:
        concern = f" Note: Long notice period of {notice} days."
    elif row["ml_years_estimate"] < 1.0 and row["ai_core_skills_count"] > 5:
        concern = " Note: Has many AI skills but limited explicit ML tenure in their role history."
        
    return sentence1 + skills_str + sentence2 + concern

top_50["reasoning"] = top_50.apply(generate_reasoning, axis=1)

print("=== 10 Example Reasonings ===\n")
for idx, r in top_50.head(10).iterrows():
    print(f"Rank: {r['final_score']:.3f} | Candidate (ID: {r['candidate_id']})")
    print(f"-> {r['reasoning']}\n")

=== 10 Example Reasonings ===

Rank: 1.000 | Candidate (ID: CAND_0000001)
-> An experienced None with 6.9 years of experience, demonstrating strong background in machine learning and relevant embeddings. Includes core AI competencies like Tailwind, NLP, Fine-tuning LLMs among 3 total ML skills. Recently active on the platform (2026-05-20).

Rank: 1.000 | Candidate (ID: CAND_0002484)
-> An experienced None with 1.3 years of experience, demonstrating strong background in machine learning and relevant embeddings. Lacks explicitly named AI/ML tools. Recently active on the platform (2026-02-22).

Rank: 1.000 | Candidate (ID: CAND_0002438)
-> An experienced None with 5.7 years of experience, demonstrating strong background in machine learning and relevant embeddings. Lacks explicitly named AI/ML tools. Highly responsive to recruiters (84% rate).

Rank: 1.000 | Candidate (ID: CAND_0002439)
-> An experienced None with 6.0 years of experience, demonstrating strong background in machine learning

### 14. Full Dataset Precomputation (Offline)
Process all 100,000 candidates. This step caches the heavy operations: feature extraction and embedding generation. We output the features to Parquet and embeddings to NPY so inference is lightweight.

In [ ]:
%pip install pyarrow fastparquet psutil

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import time
import psutil

print("=== Starting Full Dataset Precomputation ===")
start_time = time.time()

cands_path = os.path.join(raw_dir, "candidates.jsonl")
if not os.path.exists(cands_path):
    print("Warning: full candidates.jsonl not found, using sample_candidates.json for demo")
    cands_path = os.path.join(raw_dir, "sample_candidates.json")
    
is_json_array = False
try:
    with open(cands_path, 'r') as f:
        first_char = f.read(1)
    is_json_array = (first_char == '[')
except: pass

candidate_texts_full = []
candidate_ids_full = []
features_full = []

def process_candidate(cand):
    cid = cand.get("candidate_id")
    profile = cand.get("profile", {})
    history = cand.get("career_history", [])
    skills = cand.get("skills", [])
    signals = cand.get("redrob_signals", {})
    
    # 1. Text
    cand_text = build_candidate_text(cand)
    
    # 2. Features
    role_score = calc_role_title_score(profile.get("current_title"))
    company_score = calc_product_company_score(profile.get("current_company"), profile.get("current_industry"))
    
    yoe = profile.get("years_of_experience", 0)
    try: yoe_float = float(yoe) if yoe else 0.0
    except: yoe_float = 0.0
    exp_band_match = calc_experience_band_match(yoe)
    ml_years = calc_ml_years_estimate(history)
    
    ai_skills_count = sum(1 for s in skills if ai_pattern.search(s.get("name", str(s)) if isinstance(s, dict) else str(s)))
    ai_depth_score = calc_ai_skill_depth(skills)
    
    avail_score = calc_availability_score(signals)
    rel_score = calc_reliability_score(signals)
    gh_score = max(0, float(signals.get("github_activity_score", 0)))
    
    geo_score = calc_geo_fit(profile.get("location"))
    pref_work = profile.get("preferred_work_mode", "").lower()
    wm_fit = 1.0 if "hybrid" in pref_work or "flexible" in pref_work else 0.5
    notice_penalty = calc_notice_penalty(profile.get("notice_period_days"))
    
    hist_years = 0
    if history:
        try:
            earliest = min([parser.parse(r["start_date"]) for r in history if "start_date" in r])
            hist_years = (REF_DATE - earliest.replace(tzinfo=None)).days / 365.25
        except: pass
    honeypot_risk = calc_honeypot_risk(yoe_float, hist_years, skills, ml_years)
    
    return cid, cand_text, {
        "candidate_id": cid,
        "role_title_score": role_score,
        "product_company_score": company_score,
        "total_years_experience": yoe_float,
        "experience_band_match": exp_band_match,
        "ml_years_estimate": ml_years,
        "ai_core_skills_count": ai_skills_count,
        "ai_skill_depth_score": ai_depth_score,
        "availability_score": avail_score,
        "reliability_score": rel_score,
        "github_fit_score": gh_score,
        "geo_fit_score": geo_score,
        "work_mode_fit": wm_fit,
        "notice_period_penalty": notice_penalty,
        "honeypot_risk_score": honeypot_risk
    }

print(f"Streaming and feature engineering from {cands_path}...")
if is_json_array:
    with open(cands_path, "r") as f:
        data = json.load(f)
        for cand in tqdm(data, desc="Processing array"):
            cid, text, feats = process_candidate(cand)
            candidate_ids_full.append(cid)
            candidate_texts_full.append(text)
            features_full.append(feats)
else:
    # Stream with no limit
    # Tqdm without total since we don't know file length upfront
    for cand in tqdm(load_candidates(cands_path), desc="Streaming JSONL"):
        cid, text, feats = process_candidate(cand)
        candidate_ids_full.append(cid)
        candidate_texts_full.append(text)
        features_full.append(feats)

print(f"Processed {len(candidate_ids_full)} candidates.")
time_parsing = time.time() - start_time
print(f"Parsing & Feature Eng took: {time_parsing:.2f} seconds.")

# 3. Embed all text
print("Encoding all candidates (this may take a while)...")
start_emb = time.time()
if candidate_texts_full:
    # Using a larger batch size for throughput if VRAM/RAM allows
    full_embeddings = model.encode(candidate_texts_full, batch_size=128, show_progress_bar=True)
else:
    full_embeddings = np.array([])
time_emb = time.time() - start_emb
print(f"Encoding took: {time_emb:.2f} seconds.")

# 4. Save artifacts
processed_dir = os.path.join(base_dir, "data", "processed")
os.makedirs(processed_dir, exist_ok=True)

emb_path = os.path.join(processed_dir, "embeddings.npy")
ids_path = os.path.join(processed_dir, "candidate_ids.npy")
feat_path = os.path.join(processed_dir, "candidates_feather.parquet")

print(f"Saving embeddings to {emb_path}...")
np.save(emb_path, full_embeddings)

print(f"Saving candidate IDs to {ids_path}...")
np.save(ids_path, np.array(candidate_ids_full))

print(f"Saving features to {feat_path}...")
full_features_df = pd.DataFrame(features_full)
full_features_df.to_parquet(feat_path, index=False)

total_time = time.time() - start_time
process = psutil.Process(os.getpid())
mem_mb = process.memory_info().rss / (1024 * 1024)

print("=== Precomputation Complete ===")
print(f"Total Execution Time: {total_time:.2f} seconds")
print(f"Peak Memory Usage: {mem_mb:.2f} MB")

=== Starting Full Dataset Precomputation ===
Streaming and feature engineering from ../data/raw/candidates.jsonl...


Streaming JSONL: 100000it [00:17, 5845.03it/s]


Processed 100000 candidates.
Parsing & Feature Eng took: 17.12 seconds.
Encoding all candidates (this may take a while)...


Batches:   0%|          | 0/782 [00:00<?, ?it/s]

Encoding took: 863.87 seconds.
Saving embeddings to ../data/processed/embeddings.npy...
Saving candidate IDs to ../data/processed/candidate_ids.npy...
Saving features to ../data/processed/candidates_feather.parquet...
=== Precomputation Complete ===
Total Execution Time: 881.49 seconds
Peak Memory Usage: 939.25 MB
